# Ciclo de Vida del ML & Metodología CRISP-ML
## Cuaderno 3: Modelado, Evaluación e Integración - Préstamos del Campo Colombiano

Este cuaderno abarca las fases avanzadas del Ciclo de Vida del Machine Learning:
4. **Ingeniería de Modelos:** Ajuste del regresor lineal múltiple.
5. **Evaluación del Modelo:** Cálculo de métricas e inspección de supuestos.
6. **Despliegue:** Serialización y creación de la interfaz funcional de predicción.

--- 
## 1. Configuración de Entorno e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
print("Entorno configurado.")

In [ ]:
# Cargar datos
df = pd.read_csv('data/prestamos_campo_limpio.csv')
df.head()

--- 
## 2. Preparación de Variables y Partición de Datos

Separamos las características y dividimos en **80% entrenamiento** y **20% prueba**.

In [ ]:
caracteristicas = [
    'Monto_Prestamo_COP',
    'Plazo_Meses',
    'Edad_Agricultor',
    'Experiencia_Anios',
    'Ingresos_Mensuales_COP',
    'Garantia_Respaldada',
    'Subsidio_Gobierno'
]

X = df[caracteristicas]
y = df['Tasa_Interes_EA']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dimensiones entrenamiento: {X_train.shape}")

--- 
## 3. Entrenamiento de la Regresión Lineal

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)
print("¡Modelo lineal ajustado con éxito!")

--- 
## 4. Evaluación de Métricas de Desempeño

In [ ]:
y_pred = modelo.predict(X_test)

# Métricas
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=== Métricas de Desempeño del Crédito Agropecuario ===")
print(f"Coeficiente de Determinación (R²): {r2:.4f} ({r2*100:.2f}% de la varianza explicada)")
print(f"Error Absoluto Medio (MAE): {mae:.4f} puntos de tasa (E.A. %)")
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse:.4f} puntos")

### 4.1 Coeficientes (Pesos del Modelo)
Revisamos cómo afecta cada variable de manera exacta a la tasa en la ecuación final:

In [ ]:
df_coef = pd.DataFrame({
    'Característica': caracteristicas,
    'Coeficiente (Cambio en Tasa % E.A.)': modelo.coef_
})

print(f"Constante Base (Tasa inicial): {modelo.intercept_:.2f}% E.A.\n")
print(df_coef.sort_values(by='Coeficiente (Cambio en Tasa % E.A.)'))

### 4.2 Análisis de Residuos (Validación de Supuestos)

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Dispersión de residuos
sns.scatterplot(x=y_pred, y=residuos, alpha=0.6, color='#15803d', ax=axes[0])
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Gráfica de Residuos vs Predicciones', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicción de Tasa E.A. %')
axes[0].set_ylabel('Residuos')

# Histograma de residuos
sns.histplot(residuos, kde=True, color='#047857', bins=20, ax=axes[1])
axes[1].set_title('Distribución de Residuos', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residuo')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

--- 
## 5. Serialización del Modelo

Exportamos el modelo empaquetado para consumo directo en la aplicación interactiva de **Streamlit**.

In [ ]:
model_data = {
    'modelo': modelo,
    'caracteristicas': caracteristicas
}

joblib.dump(model_data, 'modelo_prestamos.pkl')
print("Modelo exportado con éxito como 'modelo_prestamos.pkl'")